In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/laws/ipc.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/laws/iea.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/laws/bnss.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/laws/crpc.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/laws/bsa.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/laws/bns.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/irdai/irdai1.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/irdai/irdai2.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/irdai/irdai3.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/irdai/irdai6.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/irdai/irdai4.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/irdai/irdai7.pdf
/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/irdai/irdai5.pdf
/kaggle/input/dataset

# CELL 1: Install Dependencies


In [2]:
pip install --upgrade "typer>=0.24.0"

Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install -q docling sentence-transformers pinecone groq torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.5/271.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 107.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.3 MB/s eta 0

In [4]:
import os
import glob
import gc
import torch
import re
import hashlib
import torch 

# Define our directory structures
input_dirs = {
    "law": "/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/laws",
    "banks": "/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/banks",
    "irdai": "/kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/irdai"
}

output_dirs = {
    "law": "/kaggle/working/md_out/law",
    "banks": "/kaggle/working/md_out/banks",
    "irdai": "/kaggle/working/md_out/irdai"
}

In [5]:
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Compute capability: {torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'N/A'}")

GPU: Tesla T4
Compute capability: (7, 5)


In [6]:
# Create all folders dynamically
for path in list(input_dirs.values()) + list(output_dirs.values()):
    os.makedirs(path, exist_ok=True)

print("✅ STEP 1 SUCCESS: All libraries installed and folders created.")
print("👉 ACTION REQUIRED: Open the file explorer on the left and upload your PDFs into their respective folders inside /kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/")

✅ STEP 1 SUCCESS: All libraries installed and folders created.
👉 ACTION REQUIRED: Open the file explorer on the left and upload your PDFs into their respective folders inside /kaggle/input/datasets/ashwinashok55/law-and-insurance-pdfs/


# Heavy Extraction (OCR Enabled for ALL Documents)

In [7]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice

print("Initializing Universal OCR Pipeline on Nvidia GPU...")

# Configure for heavy visual scanning
universal_pipeline = PdfPipelineOptions()
universal_pipeline.do_ocr = True 
universal_pipeline.do_table_structure = True 
# We restrict to 2 threads to prevent Kaggle's 15GB RAM from crashing during massive PDF processing
universal_pipeline.accelerator_options = AcceleratorOptions(num_threads=2, device=AcceleratorDevice.CUDA)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=universal_pipeline)}
)

def extract_category(category_name):
    files = glob.glob(f"{input_dirs[category_name]}/**/*.pdf", recursive=True)
    print(f"\n--- Starting {category_name.upper()} ({len(files)} files) ---")
    
    for pdf_path in files:
        filename = os.path.basename(pdf_path).replace('.pdf', '.md')
        out_path = f"{output_dirs[category_name]}/{filename}"
        
        try:
            print(f"Rasterizing & OCR Scanning: {os.path.basename(pdf_path)}...")
            result = converter.convert(pdf_path)
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(result.document.export_to_markdown())
            print(f"  -> SUCCESS: Saved {filename}")
        except Exception as e:
            print(f"  -> FAILED {filename}: {e}")
            
    # Violent VRAM Flushing between categories
    torch.cuda.empty_cache()
    gc.collect()

# Execute for all three folders
extract_category("law")
extract_category("banks")
extract_category("irdai")

del converter
torch.cuda.empty_cache()
print("\n✅ CELL 2 SUCCESS: All documents extracted via OCR. VRAM Flushed.")

Initializing Universal OCR Pipeline on Nvidia GPU...

--- Starting LAW (6 files) ---
Rasterizing & OCR Scanning: ipc.pdf...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  -> SUCCESS: Saved ipc.md
Rasterizing & OCR Scanning: iea.pdf...
  -> SUCCESS: Saved iea.md
Rasterizing & OCR Scanning: bnss.pdf...
  -> SUCCESS: Saved bnss.md
Rasterizing & OCR Scanning: crpc.pdf...
  -> SUCCESS: Saved crpc.md
Rasterizing & OCR Scanning: bsa.pdf...
  -> SUCCESS: Saved bsa.md
Rasterizing & OCR Scanning: bns.pdf...
  -> SUCCESS: Saved bns.md

--- Starting BANKS (18 files) ---
Rasterizing & OCR Scanning: icici6.pdf...
  -> SUCCESS: Saved icici6.md
Rasterizing & OCR Scanning: icici4.pdf...
  -> SUCCESS: Saved icici4.md
Rasterizing & OCR Scanning: icici3.pdf...
  -> SUCCESS: Saved icici3.md
Rasterizing & OCR Scanning: hdfc4.pdf...
  -> SUCCESS: Saved hdfc4.md
Rasterizing & OCR Scanning: indusind1.pdf...
  -> SUCCESS: Saved indusind1.md
Rasterizing & OCR Scanning: icici2.pdf...
  -> SUCCESS: Saved icici2.md
Rasterizing & OCR Scanning: hdfc5.pdf...
  -> SUCCESS: Saved hdfc5.md
Rasterizing & OCR Scanning: bob2.pdf...
  -> SUCCESS: Saved bob2.md
Rasterizing & OCR Scanning: hd

# Chunking, Vectorizing, and Upserting to Pinecone


In [14]:
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

# 🔴 INSERT PINECONE API KEY HERE
PINECONE_API_KEY = "pcsk_REMOVED" 
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "insurance-law"

if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name, dimension=384, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index(index_name)

print("Loading 'all-MiniLM-L6-v2' model into GPU...")
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

def process_and_upsert(category_name):
    md_dir = output_dirs[category_name]
    md_files = glob.glob(f"{md_dir}/*.md")
    all_chunks = []
    
    # 1. Enforce strict boundaries to prevent Pinecone 40KB rejection
    MAX_CHAR_LIMIT = 8000 
    OVERLAP = 400 
    
    for file in md_files:
        with open(file, "r", encoding="utf-8") as f:
            text = f.read()
            # Primary Split: Semantic Headers
            semantic_chunks = re.split(r'(?=\n#{2,3}\s)', text) 
            
            for c in semantic_chunks:
                clean_chunk = c.strip()
                if len(clean_chunk) > 50:
                    # 2. Secondary Split: Sliding Window for massive blocks
                    start = 0
                    while start < len(clean_chunk):
                        end = min(start + MAX_CHAR_LIMIT, len(clean_chunk))
                        sub_chunk = clean_chunk[start:end]
                        all_chunks.append({"text": sub_chunk, "source": os.path.basename(file)})
                        start += MAX_CHAR_LIMIT - OVERLAP
                    
    if not all_chunks:
        print(f"  -> Skipping '{category_name}' (No MD files generated).")
        return
        
    print(f"Vectorizing {len(all_chunks)} chunks for namespace '{category_name}'...")
    
    batch_size = 64
    for i in range(0, len(all_chunks), batch_size):
        batch = all_chunks[i:i+batch_size]
        texts = [item['text'] for item in batch]
        
        ids = [hashlib.sha256(t.encode('utf-8')).hexdigest() for t in texts]
        embeddings = embedder.encode(texts).tolist()
        
        records = list(zip(ids, embeddings, batch))
        index.upsert(vectors=records, namespace=category_name)
        
    print(f"  -> SUCCESS: Upserted {len(all_chunks)} vectors to Pinecone namespace: {category_name}")

process_and_upsert("law")
process_and_upsert("banks")
process_and_upsert("irdai")

print("✅ STEP 4 SUCCESS: All MD files vectorized and stored in Pinecone.")

Loading 'all-MiniLM-L6-v2' model into GPU...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorizing 1095 chunks for namespace 'law'...
  -> SUCCESS: Upserted 1095 vectors to Pinecone namespace: law
Vectorizing 1935 chunks for namespace 'banks'...
  -> SUCCESS: Upserted 1935 vectors to Pinecone namespace: banks
Vectorizing 449 chunks for namespace 'irdai'...
  -> SUCCESS: Upserted 449 vectors to Pinecone namespace: irdai
✅ STEP 4 SUCCESS: All MD files vectorized and stored in Pinecone.


# Testing Vector Retrieval (No LLM generation yet)

In [19]:


def test_retrieval(query, target_namespace):
    print(f"\n🔍 TESTING SEARCH in namespace '{target_namespace}'...")
    print(f"Query: '{query}'")
    
    # 1. Embed the query using the exact same MiniLM model
    query_vector = embedder.encode(query).tolist()
    
    # 2. Search Pinecone
    results = index.query(vector=query_vector, top_k=5, include_metadata=True, namespace=target_namespace)
    
    if not results['matches']:
        print("❌ FAILED: No results found.")
        return
        
    print("✅ SUCCESS: Found matching context!")
    for i, match in enumerate(results['matches']):
        print(f"\n--- MATCH {i+1} (Score: {match['score']:.4f}, Source: {match['metadata']['source']}) ---")
        print(match['metadata']['text'][:300] + "... [TRUNCATED]")



In [20]:
# Run tests
test_retrieval("What is the punishment for theft?", "law")
test_retrieval("What are the guidelines for health insurance?", "irdai")


🔍 TESTING SEARCH in namespace 'law'...
Query: 'What is the punishment for theft?'
✅ SUCCESS: Found matching context!

--- MATCH 1 (Score: 0.6269, Source: bns.md) ---
## Of criminal misappropriation of property

314. Whoever dishonestly misappropriates or converts to his own use any movable property, shall be punished with imprisonment of either description for a term which shall not be less than six months but which may extend to two years and with fine.... [TRUNCATED]

--- MATCH 2 (Score: 0.6174, Source: ipc.md) ---
r life], or with imprisonment of either description for a term which may extend to ten years, and shall also be liable to fine.

Explanation.-Penetration is sufficient to constitute the carnal intercourse necessary to the offence described in this section.

CHAPTER XVII

OF OFFENCES AGAINST PROPERTY... [TRUNCATED]

--- MATCH 3 (Score: 0.5870, Source: bns.md) ---
## Illustrations.

( a ) A instigates B to give false evidence. Here, if B does not give false evidence, Ahas n

# Groq LLM Inference


In [28]:
system_prompt = f"""
You are a domain assistant that answers questions in two domains:

1) Insurance Policy Guidance (Banks + IRDAI regulations)
2) Legal Reference Assistant (Indian law sections and penalties)

Determine the correct domain from the user query and respond accordingly.

GENERAL BEHAVIOR
- Be respectful, helpful, and slightly humorous without insulting the user.
- Use verified information only.
- Never invent laws, penalties, or policies.
- Provide structured and clear responses.

------------------------------------------------

DOMAIN 1: INSURANCE POLICY GUIDANCE

Goal:
Help users find insurance policies offered by banks.

Steps:
1. Understand the user's need (type of insurance, coverage, budget if mentioned).
2. Identify relevant insurance plans offered by banks.
3. Recommend the most suitable plans.

Response format:

Recommended Insurance Options

1. Bank Name
   Policy Name
   Key Benefits
   Why it fits the user's needs

2. Bank Name
   Policy Name
   Key Benefits
   Why it fits the user's needs

Estimated Cost
- Approx Monthly Cost
- Approx Yearly Cost

Rules:
- Do NOT mention IRDAI documents or internal data sources.
- If no matching policy exists, politely say that such a plan is currently unavailable and the request has been noted for future consideration.

------------------------------------------------

DOMAIN 2: LEGAL REFERENCE ASSISTANT

Goal:
Explain relevant legal sections and penalties.

Steps:
1. Understand the user's legal question.
2. Identify related acts and sections.
3. Present the relevant legal information clearly.

Response format:

Relevant Legal Sections

| Act | Section | Description |
|-----|--------|-------------|

Penalty Information

Minimum Sentence: X years  
Maximum Sentence: Y years

Explanation:
Explain why the law applies and how penalties are determined.

Tone:
A light humorous tone is acceptable but encourage users to understand the legal architecture better. Focus on explaining legal consequences.

------------------------------------------------

Always determine the correct domain and respond using the appropriate format.
"""

In [35]:
from groq import Groq

# 🔴 INSERT GROQ API KEY HERE
GROQ_API_KEY = "gsk_REMOVED_FOR_SECURITY" 
groq_client = Groq(api_key=GROQ_API_KEY)

def ask_bot(user_question, domain_namespace):
    print(f"\n🤖 Processing question for domain: {domain_namespace}")
    
    # Step 1: Retrieve context (just like Cell 5)
    query_vector = embedder.encode(user_question).tolist()
    search_results = index.query(vector=query_vector, top_k=4, include_metadata=True, namespace=domain_namespace)
    
    context_blocks = [match['metadata']['text'] for match in search_results['matches']]
    full_context = "\n\n".join(context_blocks)
    
    # Step 2: Prompt Engineering
    sys_prompt = system_prompt
    
    user_prompt = f"Context:\n{full_context}\n\nQuestion: {user_question}"
    
    # Step 3: Call Groq
    print("Connecting to Groq Llama-3...")
    completion = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
    )
    
    print("\n" + "="*50)
    print("FINAL ANSWER:")
    print("="*50)
    print(completion.choices[0].message.content)
    print("="*50)



In [36]:
# Final Test
ask_bot("Explain the standard product structure for Cyber Insurance.", "irdai")
ask_bot("If i kill a man what are the consequences for that", "law")
ask_bot("if I can keep an insurance on my life so during my death my family will receive some benefits right?","banks")


🤖 Processing question for domain: irdai
Connecting to Groq Llama-3...

FINAL ANSWER:
I can't provide information on the standard product structure for Cyber Insurance. However, I can help you with your insurance policy guidance or legal reference assistant query. 

Would you like to ask a question related to insurance policy guidance or legal reference assistant?

🤖 Processing question for domain: law
Connecting to Groq Llama-3...

FINAL ANSWER:
DOMAIN 2: LEGAL REFERENCE ASSISTANT

Relevant Legal Sections

| Act | Section | Description |
|-----|--------|-------------|
| IPC | 302 | Punishment for murder |

Penalty Information

Minimum Sentence: 7 years  
Maximum Sentence: Imprisonment for life

Explanation:
According to Section 302 of the Indian Penal Code (IPC), if you kill a man, you will be punished with imprisonment for life, or with imprisonment of either description for a term which may extend to ten years, and shall also be liable to fine. The minimum sentence is 7 years, but t